# 表格神经网络：分类、回归与 embedding

本 Notebook 使用 `hscredit_yyp.xlsx` 的完整有效行演示 ResNet、FTTransformer、TabM、TabPFN。
先在当前解释器安装 `pip install "hscredit[net]"`。TabPFN 首次使用会由官方库获取权重；可通过 `model_path` 指定本地权重。权重版本和许可由所选官方模型决定。

这是接口与输出验收示例；轮数较少，结果不代表最优模型效果。ResNet/FTTransformer 使用训练内验证集早停，最终测试集独立保留。

## 1. 导入与运行环境

TabPFN 示例使用 float32 推理精度，以便核对单行、批量和保存恢复的数值一致性。

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import torch
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from hscredit import ResNet, FTTransformer, TabM, TabPFN

环境 = pd.DataFrame({"组件": ["Python", "torch", "rtdl-revisiting-models", "tabpfn"],
                   "版本": [sys.version.split()[0], version("torch"), version("rtdl-revisiting-models"), version("tabpfn")]})
display(环境)

## 2. 读取真实放款数据并保留独立测试集

In [ ]:
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples/hscredit_yyp.xlsx").exists())
data = pd.read_excel(root / "examples/hscredit_yyp.xlsx")
features = ["衡枢鉴真分老客版", "近六个月非银多头机构数", "青云24", "商品类别"]
assert data["FPD"].isin([0, 1]).all()
assert np.isfinite(data["放款金额"]).all()
train_index, test_index = train_test_split(data.index, test_size=0.2, stratify=data["FPD"], random_state=42)
X_train, X_test = data.loc[train_index, features], data.loc[test_index, features]
display(pd.DataFrame({"数据集": ["全部", "训练（含内部验证）", "独立测试"],
                      "样本数": [len(data), len(train_index), len(test_index)]}))

## 3. 二分类：概率、类别与表示
分类标签为 0/1；概率列与 `classes_` 顺序一致。两种深度网络的表示就是最终线性预测头的输入。

In [ ]:
classifiers = {
    "ResNet": ResNet(d_block=32, n_blocks=2, max_epochs=15, early_stopping_rounds=5, batch_size=128, target="FPD", random_state=42),
    "FTTransformer": FTTransformer(d_block=32, n_blocks=2, max_epochs=15, early_stopping_rounds=5, batch_size=128, target="FPD", random_state=42),
    "TabM": TabM(d_block=32, n_blocks=2, k=8, max_epochs=15, early_stopping_rounds=5, batch_size=128, target="FPD", random_state=42),
    "TabPFN": TabPFN(n_estimators=2, target="FPD", random_state=42, n_jobs=1, inference_precision=torch.float32),
}
classification_rows = []
classification_outputs = {}
for name, model in classifiers.items():
    model.fit(data.loc[train_index, features + ["FPD"]])
    probability = model.predict_proba(X_test)
    prediction = model.predict(X_test)
    embedding = model.get_embeddings(X_test)
    assert probability.shape == (len(X_test), 2)
    assert prediction.shape == (len(X_test),)
    assert embedding.shape[0] == len(X_test) and embedding.ndim == 2
    assert np.isfinite(embedding).all()
    np.testing.assert_allclose(probability.sum(axis=1), 1)
    np.testing.assert_allclose(embedding, model.transform(X_test))
    np.testing.assert_allclose(model.get_embeddings(X_test.iloc[:1]), embedding[:1], atol=1e-4, rtol=1e-4)
    metrics = model.evaluate(X_test, data.loc[test_index, "FPD"], metrics=["auc", "ks", "logloss"])
    classification_rows.append({"模型": name, "测试样本数": len(X_test), "嵌入维度": embedding.shape[1], **metrics})
    classification_outputs[name] = (probability, prediction, embedding)
classification_results = pd.DataFrame(classification_rows)
display(classification_results)

In [ ]:
probability, prediction, embedding = classification_outputs["FTTransformer"]
preview = pd.DataFrame({"类别0概率": probability[:5, 0], "类别1概率": probability[:5, 1], "预测类别": prediction[:5]})
for i in range(3):
    preview[f"嵌入维度{i + 1}"] = embedding[:5, i]
display(preview)
display(classifiers["ResNet"].get_learning_curve().tail())

## 4. 回归：放款金额与表示
回归预测保留原始金额单位；`predict_proba` 不适用于回归。使用统一 `evaluate` 查询回归指标。

In [ ]:
regressors = {
    "ResNet": ResNet(objective="regression", d_block=32, n_blocks=2, max_epochs=15, early_stopping_rounds=5, batch_size=128, random_state=42),
    "FTTransformer": FTTransformer(objective="regression", d_block=32, n_blocks=2, max_epochs=15, early_stopping_rounds=5, batch_size=128, random_state=42),
    "TabM": TabM(objective="regression", d_block=32, n_blocks=2, k=8, max_epochs=15, early_stopping_rounds=5, batch_size=128, random_state=42),
    "TabPFN": TabPFN(objective="regression", n_estimators=2, random_state=42, n_jobs=1, inference_precision=torch.float32),
}
regression_rows = []
regression_outputs = {}
for name, model in regressors.items():
    model.fit(X_train, data.loc[train_index, "放款金额"])
    prediction = model.predict(X_test)
    embedding = model.get_embeddings(X_test)
    assert prediction.shape == (len(X_test),)
    assert np.isfinite(prediction).all() and np.isfinite(embedding).all()
    assert embedding.shape[0] == len(X_test) and embedding.ndim == 2
    np.testing.assert_allclose(embedding, model.transform(X_test))
    metrics = model.evaluate(X_test, data.loc[test_index, "放款金额"])
    independent_rmse = np.sqrt(np.mean((data.loc[test_index, "放款金额"].to_numpy() - prediction) ** 2))
    np.testing.assert_allclose(metrics["均方根误差"], independent_rmse)
    regression_rows.append({"模型": name, "测试样本数": len(X_test), "嵌入维度": embedding.shape[1], **metrics})
    regression_outputs[name] = (prediction, embedding)
regression_results = pd.DataFrame(regression_rows)
display(regression_results)

## 5. TabPFN 集成成员表示与保存恢复
默认按成员求均值，`aggregate=None` 保留成员轴。监督训练数据的 embedding 不等于折外特征；下游训练要在交叉验证每一折内部拟合网络。TabPFN 制品包含推理所需的训练上下文。

In [ ]:
member_embeddings = classifiers["TabPFN"].get_embeddings(X_test.iloc[:5], aggregate=None)
assert member_embeddings.ndim == 3
np.testing.assert_allclose(member_embeddings.mean(axis=0), classifiers["TabPFN"].transform(X_test.iloc[:5]))
display(pd.DataFrame({"成员数量": [member_embeddings.shape[0]], "样本数": [member_embeddings.shape[1]], "嵌入维度": [member_embeddings.shape[2]]}))

output_dir = root / ".audit_tmp" / "net-models"
output_dir.mkdir(parents=True, exist_ok=True)
for task, models in [("分类", classifiers), ("回归", regressors)]:
    for name, model in models.items():
        saved = output_dir / f"{name}_{task}.pkl"
        model.save(saved)
        restored = type(model).load(saved)
        np.testing.assert_allclose(restored.predict(X_test.iloc[:5]), model.predict(X_test.iloc[:5]))
        np.testing.assert_allclose(restored.transform(X_test.iloc[:5]), model.transform(X_test.iloc[:5]))
print("八个模型的预测和 embedding 保存恢复验证通过")

参考：[rtdl 作者实现](https://github.com/yandex-research/rtdl-revisiting-models)、[TabPFN 官方实现](https://github.com/PriorLabs/TabPFN)、[embedding 说明](https://docs.priorlabs.ai/capabilities/embeddings)。